In [ ]:
import pandas as pd
import numpy as np
import gc
import os
import joblib
import time
from sklearn.preprocessing import StandardScaler
from google.colab import drive

# 1. SETUP & PATHS
drive.mount('/content/drive', force_remount=True)
path = "/content/drive/MyDrive/MLA_Final_Project/"
base_save_path = "/content/drive/MyDrive/MLA_Final_Project/Datasets_Final/"
os.makedirs(base_save_path, exist_ok=True)

files = [
    "Monday-WorkingHours.pcap_ISCX.csv",
    "Tuesday-WorkingHours.pcap_ISCX.csv",
    "Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv",
    "Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv",
    "Friday-WorkingHours-Morning.pcap_ISCX.csv",
    "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv",
    "Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv"
]

# 2. OPTIMIZED CLEANING LOGIC (Same Logic as your Original Code)
def clean_and_optimize(df):
    df.columns = df.columns.str.strip()
    # Removing non-numeric and redundant features to prevent noise
    cols_to_drop = [
        'Flow ID', 'Source IP', 'Destination IP', 'Timestamp',
        'Bwd PSH Flags', 'Bwd URG Flags', 'Fwd Avg Bytes/Bulk',
        'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk',
        'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate'
    ]
    df.drop(columns=cols_to_drop, inplace=True, errors='ignore')
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.dropna(inplace=True)
    df = df[df["Flow Duration"] >= 0].copy()

    # Downcasting types: Essential for training 3M+ samples without crashing RAM
    df['Label'] = df['Label'].apply(lambda x: 0 if x == 'BENIGN' else 1).astype('int8')
    for col in df.select_dtypes(include=['float64']).columns:
        df[col] = df[col].astype('float32')
    for col in df.select_dtypes(include=['int64']).columns:
        df[col] = df[col].astype('int32')
    return df

# 3. PROCESSING & MERGING
processed_dfs = []
start_time = time.time()

for file in files:
    full_file_path = os.path.join(path, file)
    if os.path.exists(full_file_path):
        print(f"🔄 Processing: {file}")
        temp_df = pd.read_csv(full_file_path, low_memory=False)
        temp_df = clean_and_optimize(temp_df)
        processed_dfs.append(temp_df)
        del temp_df
        gc.collect()

full_df = pd.concat(processed_dfs, ignore_index=True)
del processed_dfs
gc.collect()

# 4. SCALING (Baseline set on Normal Monday Traffic)
print("⚖️ Scaling Data based on Normal Baseline...")
scaler = StandardScaler()
# Using only Benign traffic for the scaler baseline to define "Normalcy"
scaler.fit(full_df[full_df['Label'] == 0].drop(columns=['Label']))
joblib.dump(scaler, os.path.join(base_save_path, "scaler.save"))

# Apply transformation
y_labels = full_df['Label'].values
X_scaled = scaler.transform(full_df.drop(columns=['Label']))

# Create final DataFrame
full_final = pd.DataFrame(X_scaled, columns=full_df.columns.drop('Label'))
full_final['Label'] = y_labels

# 5. SAVE DATASETS
print("💾 Saving Datasets to Parquet...")
full_final.to_parquet(os.path.join(base_save_path, "full_data.parquet"), index=False)

# --- 🚀 PROFESSOR VERIFICATION MODULE ---
print("\n" + "="*50)
print("📊 PHASE 1: DATA AUDIT & VERIFICATION")
print("="*50)
print(f"✅ Total Runtime: {(time.time() - start_time)/60:.2f} minutes")
print(f"📈 Total Samples Processed: {len(full_final):,}")
print(f"🔢 Final Feature Count: {full_final.shape[1] - 1}")
print("\n🔍 Class Balance Check:")
print(full_final['Label'].value_counts().rename({0: 'BENIGN (Normal)', 1: 'MALICIOUS (Anomaly)'}))
print("\n📝 Sample of Cleaned & Scaled Data (First 5 Rows):")
display(full_final.head()) # This shows the actual table to your professor
print("="*50)

# Final Cleanup
del full_df, full_final, X_scaled
gc.collect()

Mounted at /content/drive
🔄 Processing: Monday-WorkingHours.pcap_ISCX.csv
🔄 Processing: Tuesday-WorkingHours.pcap_ISCX.csv
🔄 Processing: Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
🔄 Processing: Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
🔄 Processing: Friday-WorkingHours-Morning.pcap_ISCX.csv
🔄 Processing: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
🔄 Processing: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
⚖️ Scaling Data based on Normal Baseline...
💾 Saving Datasets to Parquet...

📊 PHASE 1: DATA AUDIT & VERIFICATION
✅ Total Runtime: 1.20 minutes
📈 Total Samples Processed: 2,136,376
🔢 Final Feature Count: 70

🔍 Class Balance Check:
Label
BENIGN (Normal)        1831543
MALICIOUS (Anomaly)     304833
Name: count, dtype: int64

📝 Sample of Cleaned & Scaled Data (First 5 Rows):


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,1.991080,-0.369372,-0.01031,-0.010879,-0.054232,-0.007349,-0.278851,-0.251569,-0.289266,-0.233071,...,0.003167,-0.119106,-0.1163,-0.151845,-0.090648,-0.276893,-0.090255,-0.277934,-0.264748,0
1,1.991080,-0.369372,-0.01031,-0.010879,-0.054232,-0.007349,-0.278851,-0.251569,-0.289266,-0.233071,...,0.003167,-0.119106,-0.1163,-0.151845,-0.090648,-0.276893,-0.090255,-0.277934,-0.264748,0
2,1.991080,-0.369372,-0.01031,-0.010879,-0.054232,-0.007349,-0.278851,-0.251569,-0.289266,-0.233071,...,0.003167,-0.119106,-0.1163,-0.151845,-0.090648,-0.276893,-0.090255,-0.277934,-0.264748,0
3,1.991080,-0.369372,-0.01031,-0.010879,-0.054232,-0.007349,-0.278851,-0.251569,-0.289266,-0.233071,...,0.003167,-0.119106,-0.1163,-0.151845,-0.090648,-0.276893,-0.090255,-0.277934,-0.264748,0
4,2.006041,-0.369372,-0.01031,-0.010879,-0.054232,-0.007349,-0.278851,-0.251569,-0.289266,-0.233071,...,0.003167,-0.119106,-0.1163,-0.151845,-0.090648,-0.276893,-0.090255,-0.277934,-0.264748,0


0